# 03 Gold Star Schema
Builds the Gold layer from 'workspace.cms.silver_cms_inpatient': a start schema with 'dim_hospital', 'dim_drg', and a hospital x DRG fact table, plus per-DRG median benchmarks, hospital deviations, and data quality flags (DQ-14, DQ-16, DQ-17). Starts by profiling Silver to confirm the grain and dimension keys. 

In [0]:
DESCRIBE TABLE workspace.cms.silver_cms_inpatient;

In [0]:
SELECT COUNT (*) AS n_rows
FROM workspace.cms.silver_cms_inpatient;

In [0]:
SELECT 
    COUNT (*) AS n_rows,
    COUNT (DISTINCT ccn, drg_cd ) AS n_hospital_drg_pairs

FROM workspace.cms.silver_cms_inpatient; 

In [0]:
SELECT 
    COUNT(DISTINCT ccn) AS n_hospitals,
    COUNT(DISTINCT drg_cd) AS n_drgs

FROM workspace.cms.silver_cms_inpatient;

In [0]:
SELECT COUNT(*) AS n_unstable_hospitals
FROM (
    SELECT
        ccn, 
        COUNT(DISTINCT provider_name) AS n_names,
        COUNT(DISTINCT provider_city) AS n_cities,
        COUNT(DISTINCT state_abbr) AS n_states,
        COUNT(DISTINCT zip5) AS n_zips,
        COUNT(DISTINCT ruca_code) AS n_ruca,
        COUNT(DISTINCT provider_street_addr) AS n_addrs

    FROM workspace.cms.silver_cms_inpatient
    GROUP BY ccn 
    HAVING n_names >1 OR n_cities >1 OR n_states >1 OR n_zips >1 OR n_ruca >1 OR n_addrs > 1

); 

In [0]:
SELECT
    COUNT(*) - COUNT(provider_name) AS null_names,
    COUNT(*) - COUNT(provider_city) AS null_cities,
    COUNT(*) - COUNT(state_abbr) AS null_states,
    COUNT(*) - COUNT(zip5) AS null_zips,
    COUNT(*) - COUNT(ruca_code) AS null_ruca

FROM workspace.cms.silver_cms_inpatient;

In [0]:
SELECT 
    drg_cd,
    COUNT(*) AS n_desc
FROM workspace.cms.silver_cms_inpatient
GROUP BY drg_cd
HAVING COUNT(DISTINCT drg_desc) > 1; 

In [0]:
CREATE OR REPLACE TABLE workspace.cms.gold_dim_drg AS 
SELECT DISTINCT 
    drg_cd,
    drg_desc

FROM workspace.cms.silver_cms_inpatient;

In [0]:
SELECT COUNT(*) 
FROM workspace.cms.gold_dim_drg;

In [0]:
CREATE OR REPLACE TABLE workspace.cms.gold_dim_hospital AS 
SELECT DISTINCT 
    ccn,
    provider_name,
    provider_city,
    state_abbr,
    zip5,
    ruca_code,
    provider_street_addr,
    ruca_desc,
    urban_rural,
    is_maryland

FROM workspace.cms.silver_cms_inpatient;

SELECT COUNT (*) 
FROM workspace.cms.gold_dim_hospital;

In [0]:
CREATE OR REPLACE TABLE workspace.cms.gold_fact_inpatient AS
SELECT 
ccn,
drg_cd,
tot_discharges,
avg_submitted_charge, 
avg_total_payment,
avg_medicare_payment,
tot_submitted_charge,
tot_total_payment,
tot_medicare_payment,
markup_ratio

FROM workspace.cms.silver_cms_inpatient;

SELECT COUNT(*)
FROM workspace.cms.gold_fact_inpatient;